# Library notebook — Generic helpers

**Type:** library notebook (import via `%run`; not an experiment entry point).

## Purpose

I/O, grayscale conversion, project root resolution, and display helpers for preprocessing pipelines.

**Consumers:** `01_preprocessing/0N_preprocessing_pipeline.ipynb`.


# Imports and project root

Shared imports for the preprocessing library notebooks under `01_preprocessing/00_common/`.

In [ ]:
# Inline figures in the notebook
%matplotlib inline

# ==========================================================
# IMPORTS
# ==========================================================

import io
from pathlib import Path

import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    """Walk up the directory tree until 02_dataset/ is found (Google Colab workflow)."""
    for candidate_path in [start_path, *start_path.parents]:
        if (candidate_path / "02_dataset").is_dir():
            return candidate_path
    raise FileNotFoundError(
        f"Directory 02_dataset/ not found from {start_path}."
    )

### 1. Function `convert_to_grayscale()`

Converts RGB/RGBA images to grayscale when required.

- preserves spatial dimensions (H, W);
- utiliza a course weighted luminance formula;
- if the image is already 2D, return unchanged.

In [ ]:
# ==========================================================
# 1. GRAYSCALE CONVERSION
# ==========================================================

def convert_to_grayscale(input_image):
    """
    Convert RGB/RGBA to grayscale preserving (H, W).
    """

    # If already 2D, no conversion needed
    if input_image.ndim == 2:
        return input_image

    # Extrair canais (assumindo ordem RGB)
    red_channel = input_image[:,  :, 0]
    green_channel = input_image[:,  :, 1]
    blue_channel = input_image[:,  :, 2]

    # Luminance formula (valores usados na UC)
    grayscale_image = 0.299 * red_channel + 0.587 * green_channel + 0.114 * blue_channel

    return grayscale_image

### 2. Function `ensure_uint8_manual()`

Ensures the image uses `uint8` with intensities in [0, 255].

- explicit steps: scale, round, clip, cast;
- avoids out-of-range saturation.


In [ ]:
# ==========================================================
# 2. GARANTIR FORMATO UINT8 (PASSO A PASSO)
# ==========================================================

def ensure_uint8_manual(input_image):
    """
    Convert image to uint8 no intervalo [0, 255].
    """

    # Step 1 — if already uint8, return copy
    if input_image.dtype == np.uint8:
        return input_image.copy()

    # Step 2 — use float to avoid intermediate loss
    image_float = input_image.astype(np.float64)

    # Step 3 — if values are in [0, 1], scale to [0, 255]
    maximum_intensity = image_float.max()
    if maximum_intensity <= 1.0:
        scaled_image = image_float * 255.0
    else:
        scaled_image = image_float.copy()

    # Step 4 — round to nearest integer
    rounded_image = np.round(scaled_image)

    # Step 5 — clip to [0, 255]
    clipped_image = np.clip(rounded_image, 0, 255)

    # Step 6 — convert to uint8
    input_image = clipped_image.astype(np.uint8)

    return input_image

### 3. Function `prepare_grayscale_uint8_image()`

Combines preparation steps mais frequentes: grayscale + `uint8`.

- helper reused pelos outros notebooks;
- does not alter content unnecessarily (normalises representation only).

In [ ]:
# ==========================================================
# 3. PREPARE IMAGE (GRAYSCALE + UINT8)
# ==========================================================

def prepare_grayscale_uint8_image(input_image):
    """
    Prepare image for classical processing: grayscale e uint8.
    """

    grayscale_image = convert_to_grayscale(img)
    input_image = ensure_uint8_manual(grayscale_image)

    return input_image


### 4. Function `load_image()`

Loads an image file from disk e devolve array preparado (`uint8`, grayscale).

- utiliza `matplotlib.image` (como nos worksheets);
- file_path pode ser `str` ou `Path`.

In [ ]:
# ==========================================================
# 4. LOAD IMAGE FROM PATH
# ==========================================================

def load_image(file_path):
    """
    Read image from disk e devolve uint8 em grayscale.
    """

    file_path = Path(file_path)

    if not file_path.is_file():
        raise FileNotFoundError(f"File not found: {file_path}")

    # Read with matplotlib (formato do worksheet)
    loaded_image = mpimg.imread(file_path)

    # Prepare for processing
    prepared_image = prepare_grayscale_uint8_image(loaded_image)

    return prepared_image


### 5. Function `display_images_side_by_side()`

Side-by-side display of up to 4 images (visual support during coursework).

- colormap `gray` automatic for 2D images;
- optional titles per image.

In [ ]:
# ==========================================================
# 5. SIDE-BY-SIDE IMAGE DISPLAY (UP TO 4)
# ==========================================================

def display_images_side_by_side(images_list, titles=None, show_axes=False, cmap="gray"):
    """
    Mostra entre 1 e 4 images_list na horizontal.
    """

    if not isinstance(images_list, list):
        raise ValueError("O parâmetro 'images_list' deve ser uma lista.")

    image_count = len(images_list)
    if image_count < 1 or image_count > 4:
        raise ValueError("Function accepts between 1 and 4 images.")

    plt.figure(figsize=(5 * image_count, 5))

    for indice in range(image_count):
        plt.subplot(1, image_count, indice + 1)
        current_image = images_list[indice]

        if current_image.ndim == 2:
            plt.imshow(current_image, cmap=cmap)
        else:
            plt.imshow(current_image)

        if titles is not None and indice < len(titles):
            plt.title(titles[indice])

        if not show_axes:
            plt.axis("off")

    plt.tight_layout()
    plt.show()

### 6. Function `print_image_metadata()`

Prints auxiliary array information (shape, dtype, min/max).

- useful to validate loading before filters or transforms.

In [ ]:
# ==========================================================
# 6. IMAGE METADATA (AUXILIARY)
# ==========================================================

def print_image_metadata(input_image, label="image"):
    """
    Print dimensions, dtype and intensity range.
    """

    print(f"--- Metadados: {label} ---")
    print(f"Shape: {input_image.shape}")
    print(f"Dtype: {input_image.dtype}")

    if input_image.size == 0:
        print("Empty array.")
        return

    minimum_intensity = input_image.min()
    maximum_intensity = input_image.max()
    print(f"Minimum intensity: {minimum_intensity}")
    print(f"Maximum intensity: {maximum_intensity}")


### 7. Function `create_folder_selector_widget()`

Simple text + button interface to pick a folder in Jupyter.

- optional local dataset exploration;
- returns selected path after confirmation.


In [ ]:
# ==========================================================
# 7. SELECIONAR PASTA (INTERFACE NO NOTEBOOK)
# ==========================================================

def create_folder_selector_widget(description="Folder path"):
    """
    Creates widgets for the user to pick a folder.
    Returns Path after clicking Confirm.
    """

    path_field = widgets.Text(
        description=description,
        placeholder=str(Path.cwd()),
        style={"description_width": "initial"},
    )
    confirm_button = widgets.Button(description="Confirm folder")
    area_convolution_result = widgets.Output()

    selected_path = {"valor": None}

    def on_confirm_click(_):
        with area_convolution_result:
            area_convolution_result.clear_output()
            file_path = Path(path_field.value).expanduser().resolve()
            selected_path["valor"] = file_path
            print(f"Selected folder: {file_path}")

    confirm_button.on_click(on_confirm_click)

    display(path_field, confirm_button, area_convolution_result)

    return selected_path
